# Loss Functions from Scratch: Cross-Entropy, BCE, Focal and Dice

**Difficulty**: 🟡 Medium

### Problem Statement

`F.cross_entropy` is in almost every training script, yet few people can say exactly what it computes once `weight`, `ignore_index` and `label_smoothing` are combined. Interviewers like to ask about exactly that, together with the numerical-stability questions behind every loss that takes logits:

- **Cross-entropy** with `ignore_index` (the void label 255 in semantic segmentation, the padding label -100 in captioning and LLM training), **class weights** (long-tailed data), and **label smoothing** (Inception-v3, the DeiT/ViT recipes). It takes **K-dim inputs** `(N, C, H, W)` for per-pixel classification.
- **BCE with logits** for multi-label classification, GAN discriminators and the SigLIP image–text loss. `pos_weight` trades recall against precision.
- **Sigmoid focal loss** (Lin et al., *Focal Loss for Dense Object Detection*, RetinaNet). A dense detector scores ~100k anchors per image and almost all of them are easy background. Focal loss stops these easy negatives from dominating the gradient.
- **Soft Dice loss**, the standard for binary and medical segmentation, where the foreground may be 1% of the pixels.

Implement all four from scratch with **exactly** PyTorch's / torchvision's semantics. The grader checks values **and gradients** against `F.cross_entropy` and `F.binary_cross_entropy_with_logits`, including logits as large as $10^4$.

---

### Requirements

1. **`cross_entropy(logits, target, weight=None, ignore_index=-100, label_smoothing=0.0) -> Tensor`** (0-dim) must equal `F.cross_entropy(logits, target, weight=weight, ignore_index=ignore_index, label_smoothing=label_smoothing)` with the default `reduction="mean"`.
   - `logits` is `(N, C)` or `(N, C, d1, ..., dk)`, with classes on **dim 1**. `target` is `(N,)` or `(N, d1, ..., dk)`, holding int64 class indices or `ignore_index`.
   - With $M$ = the non-ignored positions, $w_c$ = the class weights (all 1 if `weight is None`), $\log p_{i,c} = x_{i,c} - \operatorname{logsumexp}_{c'} x_{i,c'}$ and $\varepsilon$ = `label_smoothing`:
   $$\mathcal{L} = (1-\varepsilon)\,\frac{\sum_{i\in M} w_{y_i}\,(-\log p_{i,y_i})}{\sum_{i\in M} w_{y_i}} \;+\; \frac{\varepsilon}{C}\,\frac{\sum_{i\in M}\sum_{c} w_c\,(-\log p_{i,c})}{\sum_{i\in M} w_{y_i}}$$
     This is PyTorch's exact formula. The weighted mean divides by the **sum of the target weights**, not by $N$, and the smoothing term is weighted per class and shares the same denominator. Equivalently, it is the cross-entropy against the smoothed target $q = (1-\varepsilon)\,\text{onehot}(y) + \varepsilon/C$, weighted by $w_c$ and normalised by $\sum_{i \in M} w_{y_i}$.
   - Only class-index targets are required (no probability targets). If every target is ignored the result is $0/0$ = NaN, as in PyTorch (not graded).
2. **`binary_cross_entropy_with_logits(logits, targets, pos_weight=None, reduction="mean")`** must equal `F.binary_cross_entropy_with_logits`:
   $$\ell = -\big[p_w\, y\log\sigma(x) + (1-y)\log(1-\sigma(x))\big] = (1-y)\,x + \big(1 + (p_w-1)\,y\big)\,\operatorname{softplus}(-x)$$
   - `targets` may be soft (values in $[0, 1]$). `pos_weight` broadcasts against the last dimension (shape `(C,)` for multi-label). `reduction` is `"mean"`, `"sum"` or `"none"`.
   - Values and gradients must be exact for $|x|$ up to $10^4$, **and the gradient must be right at exactly $x = 0$**: a zero-initialised classifier head outputs exactly 0 on its first step.
3. **`sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0, reduction="mean")`** must equal `torchvision.ops.sigmoid_focal_loss`. `targets` are 0/1 floats with the same shape as `logits`.
   $$p = \sigma(x),\quad p_t = p\,y + (1-p)(1-y),\quad \alpha_t = \alpha\,y + (1-\alpha)(1-y),\quad \mathrm{FL} = \alpha_t\,(1-p_t)^{\gamma}\,\underbrace{(-\log p_t)}_{\text{BCE with logits}}$$
   - $\alpha$ weights the **positives** and $1-\alpha$ the negatives. `alpha < 0` disables the $\alpha_t$ factor entirely.
   - `reduction="mean"` averages over **all** elements. (RetinaNet divides the summed loss by the number of positive anchors, but that happens outside this function.)
   - With `gamma=0` and `alpha<0` it reduces to BCE-with-logits. It must stay finite for $|x|$ up to $10^4$.
4. **`dice_loss(logits, targets, eps=1.0)`** is the soft Dice loss for binary masks `(N, 1, H, W)`. Any `(N, ...)` shape works: everything except the batch dimension is summed.
   $$p = \sigma(x),\qquad \mathcal{L} = \frac{1}{N}\sum_{n=1}^{N}\left(1 - \frac{2\sum_j p_{nj}\,t_{nj} + \epsilon}{\sum_j p_{nj} + \sum_j t_{nj} + \epsilon}\right)$$
   - Dice is computed **per sample** and then averaged. `eps` appears in the numerator **and** the denominator, so an empty mask predicted empty gives a loss of 0.

---

### Constraints

- ✅ `torch.logsumexp`, `torch.logaddexp`, `F.softplus`, `F.logsigmoid`, `torch.sigmoid`, `gather`, `F.one_hot`.
- ❌ `F.cross_entropy`, `F.nll_loss`, `F.binary_cross_entropy(_with_logits)`, `nn.*Loss`, `torchvision.ops`. The grader traps the `F.*` losses. Also avoid `F.log_softmax` / `torch.log_softmax`: building log-softmax from `logsumexp` is the point.
- ✅ Vectorised: no Python loops over samples, pixels or classes.

---

<details>
  <summary>💡 Hint</summary>

  - `softmax` underflows to exactly 0 for logits about 104 below the max (float32), and `log(0) = -inf`. Can you write $\log p_c$ without ever computing $p_c$?
  - For K-dim inputs, first get the logits into an `(M, C)` matrix and the targets into an `(M,)` vector. Which axis has to move before you flatten?
  - `gather` cannot index with -100. Replace ignored targets with any valid class, then cancel their contribution with a mask.
  - $-\log\sigma(x) = \log(1 + e^{-x})$ and $-\log(1-\sigma(x)) = x + \log(1 + e^{-x})$. Which built-in computes $\log(e^a + e^b)$ stably?
  - For focal loss, reuse your BCE (with `reduction="none"`) as the $-\log p_t$ term.

</details>

---

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
torch.manual_seed(0)

# Why "from logits" matters: the naive route overflows/underflows long before 1e4.
big = torch.tensor([[1e4, 0.0, -1e4]])
print("log(softmax(x))   :", torch.log(torch.softmax(big, dim=1)))
print("x - logsumexp(x)  :", big - torch.logsumexp(big, dim=1, keepdim=True))
print("log(sigmoid(-1e4)):", torch.log(torch.sigmoid(torch.tensor(-1e4))))

# Shared test data
N, C = 16, 5
logits = torch.randn(N, C) * 3
target = torch.randint(0, C, (N,))
target_ign = target.clone()
target_ign[::4] = -100                                # every 4th target is ignored
class_w = torch.tensor([0.2, 1.0, 3.0, 0.7, 1.5])
seg_logits = torch.randn(2, C, 6, 7) * 2              # (N, C, H, W) per-pixel classification
seg_target = torch.randint(0, C, (2, 6, 7))
seg_target[0, :2] = 255                               # "void" pixels


def assert_same(got, want, what, atol=1e-5):
    assert torch.isfinite(got).all(), f"{what}: NaN/Inf"
    assert torch.allclose(got, want, atol=atol, rtol=1e-5), \
        f"{what}: got {got.detach()}, expected {want.detach()}"

In [ ]:
def cross_entropy(logits, target, weight=None, ignore_index=-100, label_smoothing=0.0):
    """F.cross_entropy with class-index targets and mean reduction.

    Args:
        logits: (N, C) or (N, C, d1, ..., dk) raw scores.
        target: (N,) or (N, d1, ..., dk) int64 class indices in [0, C), or ignore_index.
        weight: optional (C,) per-class weights.
        ignore_index (int): target value that contributes neither to the loss nor to the mean.
        label_smoothing (float): eps in [0, 1].

    Returns:
        0-dim tensor.
    """
    # TODO: (M, C) logits with classes last, (M,) targets
    # TODO: stable log-softmax via logsumexp
    # TODO: mask ignored targets; weighted NLL / sum of target weights
    # TODO: label smoothing term (weighted per class, same denominator)
    raise NotImplementedError

In [ ]:
assert_same(cross_entropy(logits, target), F.cross_entropy(logits, target), "plain")
assert_same(cross_entropy(logits, target_ign), F.cross_entropy(logits, target_ign), "ignore_index")
assert_same(cross_entropy(logits, target, weight=class_w),
            F.cross_entropy(logits, target, weight=class_w), "class weights")
assert_same(cross_entropy(logits, target_ign, weight=class_w, label_smoothing=0.2),
            F.cross_entropy(logits, target_ign, weight=class_w, label_smoothing=0.2),
            "weights + ignore + smoothing")
assert_same(cross_entropy(seg_logits, seg_target, ignore_index=255, label_smoothing=0.1),
            F.cross_entropy(seg_logits, seg_target, ignore_index=255, label_smoothing=0.1),
            "(N, C, H, W) segmentation logits")
assert_same(cross_entropy(big, torch.tensor([2])), F.cross_entropy(big, torch.tensor([2])), "1e4 logits")

x = logits.clone().requires_grad_()
cross_entropy(x, target_ign, weight=class_w, label_smoothing=0.1).backward()
x_ref = logits.clone().requires_grad_()
F.cross_entropy(x_ref, target_ign, weight=class_w, label_smoothing=0.1).backward()
assert_same(x.grad, x_ref.grad, "gradient")
print("cross_entropy matches F.cross_entropy")

In [ ]:
def binary_cross_entropy_with_logits(logits, targets, pos_weight=None, reduction="mean"):
    """F.binary_cross_entropy_with_logits, numerically stable.

    Args:
        logits: (*) raw scores.
        targets: (*) values in [0, 1] (soft labels allowed).
        pos_weight: optional tensor broadcastable to the last dim, weights the positive term.
        reduction (str): "mean" | "sum" | "none".

    Returns:
        0-dim tensor ("mean"/"sum") or a tensor shaped like logits ("none").
    """
    # TODO: write the loss with softplus(-x) = -log(sigmoid(x)); compute softplus stably
    raise NotImplementedError

In [ ]:
bx = torch.tensor([1e4, -1e4, 1e4, -1e4, 30.0, -30.0, 2.0, -1.0, 0.0])
by = torch.tensor([1.0, 1.0, 0.0, 0.0, 0.3, 0.7, 1.0, 0.0, 1.0])
pw = torch.tensor(3.0)
for kw in [dict(), dict(pos_weight=pw), dict(reduction="none"), dict(pos_weight=pw, reduction="sum")]:
    xa, xb = bx.clone().requires_grad_(), bx.clone().requires_grad_()
    got = binary_cross_entropy_with_logits(xa, by, **kw)
    want = F.binary_cross_entropy_with_logits(xb, by, **kw)
    assert_same(got, want, f"BCE {kw}")
    got.sum().backward()
    want.sum().backward()
    assert_same(xa.grad, xb.grad, f"BCE gradient {kw} (note x = 0 is in the batch)")

ml_logits, ml_targets = torch.randn(8, 4) * 4, (torch.rand(8, 4) > 0.7).float()   # multi-label
ml_pw = torch.tensor([1.0, 2.0, 5.0, 0.5])
assert_same(binary_cross_entropy_with_logits(ml_logits, ml_targets, pos_weight=ml_pw),
            F.binary_cross_entropy_with_logits(ml_logits, ml_targets, pos_weight=ml_pw), "multi-label")
print("binary_cross_entropy_with_logits matches F.binary_cross_entropy_with_logits")

In [ ]:
def sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0, reduction="mean"):
    """Sigmoid focal loss (RetinaNet), torchvision.ops.sigmoid_focal_loss semantics.

    Args:
        logits: (*) raw scores.
        targets: (*) 0/1 floats, same shape as logits.
        alpha (float): weight of positives (1 - alpha for negatives); alpha < 0 disables it.
        gamma (float): focusing parameter; gamma = 0 recovers (alpha-weighted) BCE.
        reduction (str): "mean" (over ALL elements) | "sum" | "none".
    """
    # TODO: -log(p_t) via your stable BCE (reduction="none"), modulating factor (1 - p_t)^gamma,
    #       optional alpha_t, reduction
    raise NotImplementedError

In [ ]:
fx, fy = torch.randn(6, 50) * 3, (torch.rand(6, 50) < 0.1).float()
# gamma = 0 and alpha < 0: plain BCE
assert_same(sigmoid_focal_loss(fx, fy, alpha=-1, gamma=0.0), F.binary_cross_entropy_with_logits(fx, fy),
            "gamma=0, alpha<0 must equal BCE")
# alpha weights positives, 1 - alpha negatives: a positive and a negative with the same p_t
pair_x, pair_y = torch.tensor([2.0, -2.0]), torch.tensor([1.0, 0.0])
l = sigmoid_focal_loss(pair_x, pair_y, alpha=0.25, gamma=2.0, reduction="none")
assert torch.allclose(l[0] / l[1], torch.tensor(0.25 / 0.75)), "alpha_t must be alpha for y=1, 1-alpha for y=0"
# the modulating factor down-weights easy examples far more than hard ones
easy, hard = torch.tensor([4.0]), torch.tensor([-1.0])          # both positives
ratio_easy = sigmoid_focal_loss(easy, torch.ones(1), alpha=-1) / F.binary_cross_entropy_with_logits(easy, torch.ones(1))
ratio_hard = sigmoid_focal_loss(hard, torch.ones(1), alpha=-1) / F.binary_cross_entropy_with_logits(hard, torch.ones(1))
assert ratio_easy < 1e-3 < 0.5 < ratio_hard, (ratio_easy, ratio_hard)
# finite at extreme logits, and "mean" averages over all elements
ex = torch.tensor([1e4, -1e4, 1e4, -1e4]).requires_grad_()
ey = torch.tensor([1.0, 1.0, 0.0, 0.0])
out = sigmoid_focal_loss(ex, ey, reduction="none")
out.sum().backward()
assert torch.isfinite(out).all() and torch.isfinite(ex.grad).all(), "NaN/Inf at |logits| = 1e4"
assert_same(sigmoid_focal_loss(fx, fy), sigmoid_focal_loss(fx, fy, reduction="none").mean(), "mean")
try:                                                    # optional cross-check
    from torchvision.ops import sigmoid_focal_loss as tv_focal
    assert_same(sigmoid_focal_loss(fx, fy), tv_focal(fx, fy, reduction="mean"), "vs torchvision")
    print("(matches torchvision.ops.sigmoid_focal_loss)")
except ImportError:
    pass
print("sigmoid_focal_loss tests passed")

In [ ]:
def dice_loss(logits, targets, eps=1.0):
    """Soft Dice loss for binary segmentation.

    Args:
        logits: (N, 1, H, W) (or any (N, ...)) raw mask scores.
        targets: same shape, 0/1 floats.
        eps (float): smoothing constant in numerator and denominator.

    Returns:
        0-dim tensor: mean over samples of 1 - (2*sum(p*t) + eps) / (sum(p) + sum(t) + eps).
    """
    # TODO: p = sigmoid(logits); per-sample sums over all non-batch dims; batch mean
    raise NotImplementedError

In [ ]:
# hand-computed: p = [0.5, 0.5], t = [1, 0]  ->  dice = (2*0.5 + 1) / (1 + 1 + 1) = 2/3
assert_same(dice_loss(torch.zeros(1, 1, 1, 2), torch.tensor([[[[1.0, 0.0]]]])), torch.tensor(1 / 3), "hand example")
masks = (torch.rand(3, 1, 8, 8) < 0.3).float()
masks[2] = 0                                             # an empty mask
perfect = (2 * masks - 1) * 20                           # +20 on the mask, -20 elsewhere
assert dice_loss(perfect, masks).item() < 1e-4, "perfect prediction -> ~0 (empty mask predicted empty too)"
assert dice_loss(-perfect, masks).item() > 0.6, "inverted prediction -> close to 1"
# per-sample, not global: one sample with a huge mask must not drown out one with a tiny mask
a = torch.zeros(2, 1, 4, 4)
a[0] = 1                                                 # sample 0: everything foreground
a[1, 0, 0, 0] = 1                                        # sample 1: a single pixel
pred = torch.full((2, 1, 4, 4), -20.0)
pred[0] = 20.0                                           # sample 0 right, sample 1 completely missed
assert abs(dice_loss(pred, a).item() - 0.5 * (1 - 1 / 2)) < 1e-3, "average the per-sample Dice losses"
xd = (torch.randn(2, 1, 8, 8) * 1e4).requires_grad_()
dice_loss(xd, masks[:2]).backward()
assert torch.isfinite(xd.grad).all(), "gradients must stay finite at extreme logits"
print("dice_loss tests passed")

In [ ]:
# Why focal and Dice exist: two class-imbalance stories.
torch.manual_seed(0)
# 1) Dense detection: 10,000 anchors, 1% positives. The classifier already rejects most
#    background confidently (easy negatives), but there are SO many of them.
n_pos, n_neg = 100, 9900
x = torch.cat([torch.randn(n_pos) * 1.5,                 # positives: still uncertain (hard)
               torch.randn(n_neg) * 1.0 - 4.0])          # negatives: mostly easy, p ~ 0.02
y = torch.cat([torch.ones(n_pos), torch.zeros(n_neg)])
for name, per_elem in [("BCE", binary_cross_entropy_with_logits(x, y, reduction="none")),
                       ("focal (a=.25, g=2)", sigmoid_focal_loss(x, y, reduction="none"))]:
    share = per_elem[n_pos:].sum() / per_elem.sum()
    print(f"{name:>20}: easy-negative share of the total loss = {share:.1%}")

# 2) Segmentation with a 2% foreground: predicting "all background" looks fine to BCE.
mask = torch.zeros(4, 1, 32, 32)
mask[:, :, 10:14, 10:15] = 1                             # 20 of 1024 pixels
all_background = torch.full_like(mask, -4.0)
print(f"all-background prediction: BCE = {binary_cross_entropy_with_logits(all_background, mask):.3f}"
      f"   Dice loss = {dice_loss(all_background, mask):.3f}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("losses-from-scratch", cross_entropy=cross_entropy,
      binary_cross_entropy_with_logits=binary_cross_entropy_with_logits,
      sigmoid_focal_loss=sigmoid_focal_loss, dice_loss=dice_loss)